<a href="https://colab.research.google.com/github/UdayKajana/udaykajana-codeforge/blob/artificial_intelligence/playground/dspy_handoff.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Installations and Imports

In [1]:
!pip install openai-agents dspy
AZURE_API_KEY     = ""
AZURE_ENDPOINT    = ""
AZURE_API_VERSION = ""
AZURE_DEPLOYMENT  = ""

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 843.0/843.0 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.0/331.0 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.5/146.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.0/17.0 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 15.8 MB/s eta 0:00:00
  Attempting uninstall: typeguard
    Found existing installation: typeguard 4.5.2
    Uninstalling typeguard-4.5.2:
      Successfully uninstalled typeguard-4.5.2


## Email Extraction

In [4]:
# Importing emails
import json
emails = dict()
with open("/content/bidding_emails.json", encoding="utf-8") as f:
    emails = json.load(f)
for key, body in emails.items():
    print(key, "→", body[:200])


email_01 → FROM: sandeep.rao@delta-infra.com
TO: bids@acmecorp.com
DATE: Monday, May 19, 2026 10:32 AM
SUBJECT: Welding Robots – Looking for a Good Vendor, Thought of You Guys

Hi team,

Hope you're all doing we
email_02 → FROM: supply@greenfield-auto.com
TO: vendors@acmemotors.com
DATE: Tuesday, May 20, 2026 2:15 PM
SUBJECT: RFQ – EV Battery Packs for Q3 Production Run

Dear Vendor,

Good afternoon. We are reaching out
email_03 → FROM: it.buying@novatechsols.com
TO: sales@acmecorp.com
DATE: Wednesday, May 21, 2026 9:48 AM
SUBJECT: Bulk Laptop Procurement – Bid Invitation (Urgent)

Hello,

I hope this finds you well. We are exp
email_04 → FROM: james.whitfield@metro-display.co.uk
TO: procurement.bids@acmecorp.com
DATE: Thursday, May 22, 2026 11:05 AM
SUBJECT: OLED Panel RFQ – 50K Units – Q3/Q4 Delivery

Hi,

We spoke briefly at Display
email_05 → FROM: admin.purchase@globalspace-offices.com
TO: furniture.bids@acmecorp.com
DATE: Friday, May 23, 2026 3:22 PM
SUBJECT: Office Furniture – 2

## Agent-Agent With DSPy - General Usecase

In [ ]:
import dspy
from openai import AsyncAzureOpenAI
from agents import set_default_openai_client, set_default_openai_api
import asyncio
from dataclasses import dataclass
from agents import Agent, Runner, ModelSettings, set_tracing_disabled
set_tracing_disabled(True)


client = AsyncAzureOpenAI(
    api_key        = AZURE_API_KEY,
    azure_endpoint = AZURE_ENDPOINT,
    api_version    = AZURE_API_VERSION,
)
set_default_openai_client(client)
set_default_openai_api("chat_completions")

lm = dspy.LM(
    model       = f"azure/{AZURE_DEPLOYMENT}",
    api_key     = AZURE_API_KEY,
    api_base    = AZURE_ENDPOINT,
    api_version = AZURE_API_VERSION,
    max_tokens  = 1024,
    temperature = 0.0,   # ← deterministic skeleton generation
)
dspy.configure(lm=lm)

# ── DSPy: raw prompt → JSON skeleton with empty values ───────────────────────

class BuildSkeleton(dspy.Signature):
    """
    Given a user request, produce a JSON skeleton.
    Rules:
      - Keys must be short, descriptive snake_case strings.
      - Fill ONLY the keys you can infer directly from the request
        (e.g. name, category, type). Leave everything else as "".
      - Use plenty of keys covering: identity, geography, economy,
        history, culture, politics, notable_facts, and any topic-specific fields.
      - Output raw JSON only — no markdown fences, no explanation.
        make sure you provide all possible keys as possible, but make sure it should neither lengthy json not short json, it should address all aspects of the request.
        arrange the keys in catogories wise, like finance, history, specifications, desclaimers, hardware_requests etc.
    """
    user_request:  str = dspy.InputField(desc="The raw user request, e.g. 'Tell me about China'")
    json_skeleton: str = dspy.OutputField(desc="Raw JSON object with snake_case keys and empty string values, except for fields directly inferable from the request")

def dspy_transform(prompt: str) -> str:
    return dspy.Predict(BuildSkeleton)(user_request=prompt).json_skeleton

# ── SubAgent: receives skeleton → fills every value ──────────────────────────

sub_agent = Agent(
    name="SubAgent",
    model=AZURE_DEPLOYMENT,
    model_settings=ModelSettings(
        max_tokens  = 2048,
        temperature = 0.0,   # ← fixed typo: was 'temparature'
    ),
    instructions=(
        "You are a data enrichment agent. "
        "You receive a JSON object where most values are empty strings. "
        "Fill in EVERY empty value with accurate, factual information. "
        "Do not remove any keys. Do not add prose. "
        "Return only the completed JSON — no markdown fences, no explanation."
    ),
)

# ── Pipeline ──────────────────────────────────────────────────────────────────

async def run_pipeline(user_prompt: str) -> str:
    print(f"[User]       {user_prompt}\n")

    # Step 1 — DSPy builds the skeleton
    skeleton = dspy_transform(user_prompt.strip())
    print(f"[DSPy]       {skeleton}\n")

    # Step 2 — SubAgent fills every value
    result = await Runner.run(sub_agent, input=skeleton)
    print(f"[SubAgent]   {result.final_output}")

    return result.final_output

await run_pipeline("Write about china")

[User]       Write about china

[DSPy]       {
  "identity": {
    "name": "China",
    "official_name": "People's Republic of China",
    "type": "country",
    "region": "East Asia"
  },
  "geography": {
    "area": "",
    "population": "",
    "capital": "Beijing",
    "major_cities": "",
    "climate": "",
    "landmarks": "",
    "borders": ""
  },
  "economy": {
    "currency": "Renminbi (Yuan)",
    "gdp": "",
    "major_industries": "",
    "trade_partners": "",
    "economic_system": "Socialist market economy"
  },
  "history": {
    "ancient_history": "",
    "modern_history": "",
    "dynasties": "",
    "key_events": "",
    "revolution": "",
    "founding_date": "October 1, 1949"
  },
  "culture": {
    "language": "Mandarin Chinese",
    "religions": "",
    "traditions": "",
    "festivals": "",
    "cuisine": "",
    "arts": "",
    "sports": ""
  },
  "politics": {
    "government_type": "Communist one-party state",
    "current_leader": "",
    "political_parties": "

'{\n  "identity": {\n    "name": "China",\n    "official_name": "People\'s Republic of China",\n    "type": "country",\n    "region": "East Asia"\n  },\n  "geography": {\n    "area": "9,596,961 square kilometers",\n    "population": "1,425,893,465 (2023 estimate)",\n    "capital": "Beijing",\n    "major_cities": "Shanghai, Guangzhou, Shenzhen, Chongqing, Tianjin",\n    "climate": "Varies from subarctic in the north to tropical in the south",\n    "landmarks": "Great Wall of China, Forbidden City, Terracotta Army, Potala Palace, Zhangjiajie National Forest Park",\n    "borders": "14 countries: Afghanistan, Bhutan, India, Kazakhstan, Kyrgyzstan, Laos, Mongolia, Myanmar, Nepal, North Korea, Pakistan, Russia, Tajikistan, Vietnam"\n  },\n  "economy": {\n    "currency": "Renminbi (Yuan)",\n    "gdp": "18.32 trillion USD (2022)",\n    "major_industries": "Manufacturing, technology, agriculture, mining, textiles, electronics",\n    "trade_partners": "United States, European Union, Japan, South

## Without DSPy - Email case

In [15]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║         PROCUREMENT EMAIL PIPELINE — No DSPy, Prompt-Config Driven          ║
║                                                                              ║
║  All system prompts and agent settings live in prompts.json.                 ║
║  The pipeline loads them at startup — no hardcoded prompt strings here.      ║
║                                                                              ║
║  prompts.json structure:                                                     ║
║    {                                                                         ║
║      "agent_settings": {                                                     ║
║        "<agent_key>": { name, max_tokens, temperature }                      ║
║      },                                                                      ║
║      "system_prompts": {                                                     ║
║        "<agent_key>": "<full system prompt string>"                          ║
║      }                                                                       ║
║    }                                                                         ║
║                                                                              ║
║  Agent keys (must exist in both sections of prompts.json):                  ║
║    forensic_extractor  → Agent1: reads email, builds structured skeleton     ║
║    structural_auditor  → Agent2: audits structure, quarantines noise         ║
║    enrichment_engine   → Agent3: fills values, resolves, validates           ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

import json
import asyncio
from pathlib import Path
from openai import AsyncAzureOpenAI
from agents import (
    Agent, Runner, ModelSettings,
    set_default_openai_client, set_default_openai_api,
    set_tracing_disabled,
)
import nest_asyncio
nest_asyncio.apply()
set_tracing_disabled(True)


# ──────────────────────────────────────────────────────────────────────────────
# Azure credentials
# ──────────────────────────────────────────────────────────────────────────────

azure_client = AsyncAzureOpenAI(
    api_key        = AZURE_API_KEY,
    azure_endpoint = AZURE_ENDPOINT,
    api_version    = AZURE_API_VERSION,
)
set_default_openai_client(azure_client)
set_default_openai_api("chat_completions")


# ──────────────────────────────────────────────────────────────────────────────
# Load prompts config from JSON
# ──────────────────────────────────────────────────────────────────────────────
PROMPTS_FILE = "/content/prompts.json"

def load_prompts(path: Path) -> dict:
    """
    Load and validate prompts.json.
    Returns the full config dict.
    Raises KeyError if any expected agent key is missing from either section.
    """
    with open(path, encoding="utf-8") as f:
        cfg = json.load(f)

    required_keys   = {"forensic_extractor", "structural_auditor", "enrichment_engine"}
    prompts_keys    = set(cfg.get("system_prompts", {}).keys())
    settings_keys   = set(cfg.get("agent_settings", {}).keys())

    missing_prompts  = required_keys - prompts_keys
    missing_settings = required_keys - settings_keys

    if missing_prompts:
        raise KeyError(
            f"prompts.json is missing these keys in 'system_prompts': {missing_prompts}"
        )
    if missing_settings:
        raise KeyError(
            f"prompts.json is missing these keys in 'agent_settings': {missing_settings}"
        )

    return cfg


cfg      = load_prompts(PROMPTS_FILE)
PROMPTS  = cfg["system_prompts"]    # agent_key → prompt string
SETTINGS = cfg["agent_settings"]    # agent_key → { name, max_tokens, temperature }

print(f"[Config] Loaded prompts.json — {len(PROMPTS)} agent prompts registered.")
for key in PROMPTS:
    s = SETTINGS[key]
    print(
        f"  [{key}] name={s['name']} | "
        f"max_tokens={s['max_tokens']} | "
        f"temperature={s['temperature']} | "
        f"prompt_length={len(PROMPTS[key])} chars"
    )


# ──────────────────────────────────────────────────────────────────────────────
# Build agents from config
# Each agent reads its system prompt and settings from the JSON dict by key.
# ──────────────────────────────────────────────────────────────────────────────

def build_agent(key: str) -> Agent:
    """Construct an Agent using settings and prompt from the loaded config."""
    s = SETTINGS[key]
    return Agent(
        name           = s["name"],
        model          = AZURE_DEPLOYMENT,
        model_settings = ModelSettings(
            max_tokens  = s["max_tokens"],
            temperature = s["temperature"],
        ),
        instructions   = PROMPTS[key],   # ← prompt loaded from prompts.json[key]
    )


agent1 = build_agent("forensic_extractor")   # reads email → JSON skeleton
agent2 = build_agent("structural_auditor")   # audits structure → gaps + quarantine
agent3 = build_agent("enrichment_engine")    # fills + resolves + validates → final JSON


# ──────────────────────────────────────────────────────────────────────────────
# Helpers
# ──────────────────────────────────────────────────────────────────────────────

def strip_fences(text: str) -> str:
    """Remove accidental markdown code fences that models sometimes emit."""
    return (
        text.strip()
        .removeprefix("```json")
        .removeprefix("```")
        .removesuffix("```")
        .strip()
    )


# ──────────────────────────────────────────────────────────────────────────────
# Pipeline runner
# ──────────────────────────────────────────────────────────────────────────────

async def run_pipeline(email_key: str, email_text: str) -> dict:
    print(f"\n{'═'*64}")
    print(f"  Processing: {email_key}")
    print(f"{'═'*64}")

    # ── Agent 1 — Forensic Extractor ─────────────────────────────────────────
    # Prompt key: "forensic_extractor"
    # Job: extract only what is stated; build domain-specific tag_list skeleton
    print(f"\n[Agent1 — {SETTINGS['forensic_extractor']['name']}] Reading email...")
    result1       = await Runner.run(agent1, input=email_text)
    agent1_output = result1.final_output
    print(f"[Agent1 done] {len(agent1_output)} chars")

    # ── Agent 2 — Structural Auditor ─────────────────────────────────────────
    # Prompt key: "structural_auditor"
    # Job: add missing fields as ""; quarantine duplicates/irrelevant/ambiguous;
    #      write agent3_briefing in audit_report
    print(f"[Agent2 — {SETTINGS['structural_auditor']['name']}] Auditing structure...")
    result2       = await Runner.run(agent2, input=agent1_output)
    agent2_output = result2.final_output
    print(f"[Agent2 done] {len(agent2_output)} chars")

    # ── Agent 3 — Enrichment Engine ──────────────────────────────────────────
    # Prompt key: "enrichment_engine"
    # Job: fill all "" values; resolve review_section; validate; return final JSON
    print(f"[Agent3 — {SETTINGS['enrichment_engine']['name']}] Enriching and validating...")
    result3      = await Runner.run(agent3, input=agent2_output)
    final_output = result3.final_output
    print(f"[Agent3 done] {len(final_output)} chars")

    # ── Parse final output ────────────────────────────────────────────────────
    try:
        parsed = json.loads(strip_fences(final_output))
    except json.JSONDecodeError as e:
        print(f"[WARNING] JSON parse failed for '{email_key}': {e}")
        parsed = {"raw_output": final_output, "parse_error": str(e)}

    # ── Print summary ─────────────────────────────────────────────────────────
    print(f"\n[FINAL — {email_key}]")
    summary = json.dumps(parsed, indent=2, ensure_ascii=False)
    print(summary)

    return {
        "email_key"     : email_key,
        "agent1_output" : agent1_output,   # raw Agent1 JSON string
        "agent2_output" : agent2_output,   # raw Agent2 JSON string (with audit_report)
        "final"         : parsed,          # parsed final dict
    }


async def run_all(emails: dict) -> list:
    results = []
    for key, body in emails.items():
        result = await run_pipeline(key, body)
        results.append(result)
    return results


# ──────────────────────────────────────────────────────────────────────────────
# Entry point
# ──────────────────────────────────────────────────────────────────────────────

emails = {}
with open("/content/bidding_emails.json", encoding="utf-8") as f:
    emails = json.load(f)

print(f"\nLoaded {len(emails)} emails. Starting pipeline...\n")
results = await run_all(emails)

print(f"\n{'═'*64}")
print(f"  Pipeline complete. {len(results)} emails processed.")
print(f"{'═'*64}")

[Config] Loaded prompts.json — 3 agent prompts registered.
  [forensic_extractor] name=ForensicExtractorAgent | max_tokens=2500 | temperature=0.0 | prompt_length=6964 chars
  [structural_auditor] name=StructuralAuditorAgent | max_tokens=3000 | temperature=0.0 | prompt_length=5880 chars
  [enrichment_engine] name=EnrichmentDecisionAgent | max_tokens=4000 | temperature=0.0 | prompt_length=5992 chars

Loaded 10 emails. Starting pipeline...


════════════════════════════════════════════════════════════════
  Processing: email_01
════════════════════════════════════════════════════════════════

[Agent1 — ForensicExtractorAgent] Reading email...
[Agent1 done] 3293 chars
[Agent2 — StructuralAuditorAgent] Auditing structure...
[Agent2 done] 5976 chars
[Agent3 — EnrichmentDecisionAgent] Enriching and validating...
[Agent3 done] 6894 chars

[FINAL — email_01]
{
  "email_meta": {
    "from_address": "sandeep.rao@delta-infra.com",
    "to_address": "bids@acmecorp.com",
    "date_sent": "Monday, Ma

## With DSPy - Email case

In [16]:
"""
Pipeline: Agent1 → DSPy → Agent2

Agent1  — reads raw email, extracts only what is explicitly stated, flags all standard-but-missing fields
DSPy    — structural auditor: adds missing domain-standard fields, quarantines noise with typed instructions
Agent2  — enrichment + decision engine: fills every gap, resolves quarantine, emits final clean JSON
"""

import json
import asyncio
import dspy
from openai import AsyncAzureOpenAI
from agents import (
    Agent, Runner, ModelSettings,
    set_default_openai_client, set_default_openai_api,
    set_tracing_disabled,
)
import nest_asyncio
nest_asyncio.apply()
set_tracing_disabled(True)


# ─────────────────────────────────────────────────────────────────────────────
# Azure + DSPy Setup
# ─────────────────────────────────────────────────────────────────────────────

azure_client = AsyncAzureOpenAI(
    api_key        = AZURE_API_KEY,
    azure_endpoint = AZURE_ENDPOINT,
    api_version    = AZURE_API_VERSION,
)
set_default_openai_client(azure_client)
set_default_openai_api("chat_completions")

lm = dspy.LM(
    model       = f"azure/{AZURE_DEPLOYMENT}",
    api_key     = AZURE_API_KEY,
    api_base    = AZURE_ENDPOINT,
    api_version = AZURE_API_VERSION,
    max_tokens  = 2048,
    temperature = 0.0,
)
dspy.configure(lm=lm)


# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — Agent 1: Procurement Email Extractor
#
# ROLE: A forensic extractor. It reads the raw email and produces a structured
# JSON with two clear categories:
#   (a) what the email actually says → goes into extracted_values
#   (b) what a standard procurement request of this type needs → goes into tag_list
#       with the sender's values copied over where available, "" where not.
#
# CRITICAL RULE: Agent1 NEVER invents, infers, or assumes values.
# Its job is extraction and classification, not enrichment.
# ─────────────────────────────────────────────────────────────────────────────

AGENT1_INSTRUCTIONS = """
You are a forensic procurement email extractor. Your only job is to read a raw email and produce a structured JSON.

You operate in TWO modes simultaneously:
  MODE A — EXTRACT: Capture every concrete fact the sender actually wrote.
  MODE B — INVENTORY: List every field that a standard procurement request of this type should contain.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTPUT STRUCTURE — return exactly these 5 top-level keys:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. "email_meta"
   Keys: from_address, to_address, date_sent, subject
   Rule: Fill from email headers only. Use null if not present.

2. "context"
   Keys:
     domain          → industry/sector (e.g. "manufacturing", "IT", "logistics", "raw_materials")
     request_type    → category (e.g. "rfq", "procurement_bid", "contract_renewal", "vendor_registration")
     summary         → 1-2 sentence plain English description of exactly what the sender is asking for
     tone            → one word (e.g. "formal", "urgent", "casual", "technical")
     relationship     → one of: "first_contact" | "existing_vendor" | "referred" | "unknown"
   Rule: Derive these by reading the email. These are YOUR classifications, not the sender's words.

3. "extracted_values"
   Capture every concrete value the sender stated:
     - Product/service names, model numbers, specifications
     - Quantities, units of measure
     - Prices, budgets, currencies
     - Dates (order date, delivery date, deadline, contract period)
     - Locations (delivery, installation, origin)
     - Contact details (name, phone, email)
     - Brand preferences, exclusions
     - Certifications or compliance mentioned
     - Payment terms, milestones, phasing
     - Any other specific numbers, names, or facts
   Rule: Use the sender's exact wording and numbers. No paraphrasing. Null if not mentioned.

4. "tag_list"
   Purpose: A complete checklist of ALL fields this type of procurement request requires.
   Construction rules:
     (a) Start from domain + request_type to determine the required field set.
     (b) For every required field:
         - If the sender mentioned a value: copy it from extracted_values.
         - If the sender did NOT mention it: set value to "" (empty string, NOT null).
     (c) Think across these categories for this domain:
         IDENTIFICATION  → ref_number, rfq_id, po_number, contract_id
         PRODUCT/SERVICE → item_name, item_description, part_number, quantity, unit_of_measure, brand, model
         COMMERCIAL      → unit_price, total_budget, currency, incoterms, payment_terms, payment_schedule
         TIMELINE        → rfq_issue_date, bid_submission_deadline, delivery_date, contract_start, contract_end
         LOGISTICS       → delivery_location, shipping_method, packaging_requirements, lead_time_days
         COMPLIANCE      → certifications_required, regulatory_standards, country_of_origin, mtl_traceability
         VENDOR          → vendor_name, vendor_registration_number, vendor_category, past_relationship
         EVALUATION      → evaluation_criteria, scoring_weights, min_qty_threshold, sample_required
         DOMAIN-SPECIFIC → add fields specific to the detected domain (see examples below)
     (d) Domain-specific field examples:
         IT procurement   → os_preference, support_sla, warranty_years, asset_tagging_required, cybersecurity_standard
         Fleet/EV         → vehicle_type, payload_kg, range_km, charging_infrastructure, fleet_size
         Raw materials    → material_grade, chemical_composition, mtr_required, rejection_tolerance, batch_size
         Construction     → site_location, project_phase, structural_specs, safety_standards, insurance_required
         Pharma/Medical   → regulatory_body, shelf_life, cold_chain, gmp_certification, batch_traceability
   Rule: "" means "this field exists but the sender left it blank". It is NOT optional — every standard field must appear.

5. "sender_notes"
   An array of strings. Capture:
     - Side remarks or off-topic comments from the sender
     - Contradictions or inconsistencies within the email
     - Items the sender said they will send later
     - Anything ambiguous that a procurement officer would flag for clarification
   Rule: Quote the sender's words where possible. Empty array [] if nothing to note.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STRICT RULES:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Output raw JSON only. No markdown fences. No prose before or after the JSON.
• Do NOT invent, infer, or assume any value in extracted_values. Only what is in the email.
• Every key in tag_list must be snake_case, lowercase, no spaces.
• The JSON must be valid and parseable. Test it mentally before outputting.
• Do not repeat the same information across sections. extracted_values is the source; tag_list references it.
"""

agent1 = Agent(
    name           = "ProcurementExtractorAgent",
    model          = AZURE_DEPLOYMENT,
    model_settings = ModelSettings(max_tokens=2048, temperature=0.0),
    instructions   = AGENT1_INSTRUCTIONS,
)


# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — DSPy: Structural Auditor
#
# ROLE: A structural auditor that reviews Agent1's output for completeness
# and cleanliness. It does NOT fill values. It only:
#   (a) Adds standard fields that Agent1 missed entirely
#   (b) Moves problematic fields into a quarantine zone with typed instructions
#   (c) Writes a precise briefing for Agent2
#
# DSPy is the quality gate between extraction and enrichment.
# ─────────────────────────────────────────────────────────────────────────────

class StructuralAudit(dspy.Signature):
    """
    You are a procurement JSON auditor. You receive a structured extraction from Agent1.
    Your job is STRUCTURAL ONLY. You do not fill values. You reorganize and annotate.

    INPUT: A JSON with keys: email_meta, context, extracted_values, tag_list, sender_notes.

    YOU MUST DO EXACTLY THREE THINGS:

    ── PART A: Close field gaps ─────────────────────────────────────────────
    Compare tag_list against the domain (context.domain) and request_type (context.request_type).
    Identify fields that are ABSENT from tag_list but are standard requirements for this domain.
    Ask yourself: "If a senior procurement officer reviewed this request, what field would they
    immediately notice is missing?"
    Add each missing field to tag_list with value "" (empty string).
    Do NOT modify or remove any existing tag_list entries.
    Do NOT add fields that are already present under a different name.

    ── PART B: Quarantine problematic fields ────────────────────────────────
    Review every key in tag_list. Move a key to "review_section" if it is:
      TYPE 1 — DUPLICATE: same data already captured under another key name
               → instruction format: "DUPLICATE of <other_key> — Agent2: merge value into <other_key> and drop this key"
      TYPE 2 — IRRELEVANT: clearly not applicable to this domain or request type
               → instruction format: "IRRELEVANT to <domain> <request_type> — Agent2: drop unless sender specifically requested"
      TYPE 3 — AMBIGUOUS: too vague to be actionable or bidable
               → instruction format: "AMBIGUOUS — Agent2: rephrase to '<suggested_specific_name>' or drop if unresolvable"
      TYPE 4 — MISPLACED: belongs in a different section (e.g. a process note in a value field)
               → instruction format: "MISPLACED — Agent2: move to sender_notes or email_meta as appropriate"
    For each moved key, add a companion key named <original_key>__instruction with the typed instruction string above.
    Leave both the value AND the instruction in review_section.

    ── PART C: Write Agent2's briefing ─────────────────────────────────────
    Add a top-level "dspy_audit" object with:
      "gaps_added": [list of key names you added in Part A]
      "quarantined": [list of key names moved to review_section in Part B]
      "completeness_score": integer 0-100 estimating how complete tag_list was before your additions
      "agent2_briefing": a structured string with exactly these sub-sections:
        FILL: "Fill all empty-string values in tag_list using domain knowledge, market standards,
               and reasonable defaults. Compute derived fields (e.g. total = qty × unit_price)."
        RESOLVE: "For each key in review_section, execute the typed instruction in its __instruction sibling.
                  Document each decision in review_decisions."
        VALIDATE: "After filling and resolving, run a feasibility and risk check."
        OUTPUT: "Return final clean JSON. Remove review_section, all __instruction keys, and dspy_audit."

    OUTPUT RULES:
    • Return raw JSON only. No markdown fences. No prose outside the JSON.
    • The output must be valid, parseable JSON.
    • Preserve all existing keys. Only add or move — never delete without moving to review_section.
    """
    agent1_json   : str = dspy.InputField(
        desc="Complete JSON from Agent1 with keys: email_meta, context, extracted_values, tag_list, sender_notes"
    )
    audited_json  : str = dspy.OutputField(
        desc=(
            "Modified JSON with: "
            "(1) tag_list enriched with missing-field keys set to empty string, "
            "(2) problematic keys moved to review_section with typed __instruction companions, "
            "(3) dspy_audit section added with gaps_added, quarantined, completeness_score, agent2_briefing."
        )
    )


def dspy_audit(agent1_output: str) -> str:
    return dspy.Predict(StructuralAudit)(agent1_json=agent1_output).audited_json


# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — Agent 2: Enrichment & Decision Engine
#
# ROLE: The final intelligent layer. It receives the audited JSON and must:
#   (a) Fill EVERY empty field in tag_list with accurate, domain-appropriate data
#   (b) Process EVERY quarantine item and execute its typed instruction
#   (c) Run a validation pass for feasibility and risk
#   (d) Return one final clean, publication-ready JSON
#
# Agent2 has the authority to make decisions. It must justify each one.
# ─────────────────────────────────────────────────────────────────────────────

AGENT2_INSTRUCTIONS = """
You are a senior procurement enrichment and decision agent. You receive a JSON that has been
extracted by Agent1 and audited by a structural AI. Your job is to complete it, resolve all
open questions, and produce the final clean procurement record.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BEFORE ANYTHING ELSE — READ THIS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. Read "dspy_audit.agent2_briefing" in full. This is your instruction set.
2. Note "dspy_audit.gaps_added" — these are fields just added and need filling.
3. Note "dspy_audit.quarantined" — these are fields waiting for your decision.
4. DO NOT change email_meta, context, or extracted_values. They are finalized.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TASK 1: FILL tag_list (highest priority)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
For every key in tag_list where value is "" (empty string):
  • Use your domain knowledge to supply the most accurate, specific value.
  • Prioritize: (1) values derivable from other filled fields, (2) industry standards and norms,
    (3) market-typical defaults for this domain and region.
  • For computed fields: calculate them (e.g. total_value = quantity × unit_price).
  • For date fields: if a relative date can be inferred (e.g. "delivery in 6 weeks"), compute
    an absolute date from the email's date_sent.
  • For compliance/certification fields: list the standard certifications required for this domain
    in the likely region of the sender.
  • For financial fields: use current market rate ranges as defaults, not made-up numbers.
    State the basis (e.g. "industry average Q1 2025").
  • If a field is genuinely unknowable without sender input, set it to "__NEEDS_SENDER_INPUT__"
    (not "" — mark it explicitly so it surfaces in validation).

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TASK 2: RESOLVE review_section
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
For each key in review_section, read its <key>__instruction and execute it:
  • DUPLICATE instruction → merge the value into the correct tag_list key. Drop the duplicate.
  • IRRELEVANT instruction → drop it unless there is a compelling reason to keep it.
  • AMBIGUOUS instruction → apply the suggested rename and fill the value. If unresolvable, drop.
  • MISPLACED instruction → move to the appropriate section (sender_notes or email_meta).

Document every decision in a "review_decisions" section:
  Format: { "<original_key>": { "action": "merged|dropped|renamed|moved", "reason": "...", "destination": "..." } }

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TASK 3: VALIDATE and flag
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Add a "validation" section with:

  "budget_feasibility":
    Compare the stated budget (or derived total) against market rates for this request.
    Output: "feasible" | "tight" | "likely_insufficient" | "unclear"
    Include a one-sentence justification.

  "missing_critical_fields":
    List every tag_list key still set to "__NEEDS_SENDER_INPUT__" after Task 1.
    These are blockers — the procurement team cannot proceed without them.

  "timeline_assessment":
    Is the deadline realistic given lead times, production, and shipping for this domain?
    Output: "realistic" | "tight" | "unrealistic" | "unclear"
    Include a one-sentence justification.

  "risk_flags":
    Array of strings. Flag any of:
      - Budget/quantity mismatch
      - Contradictions between email fields
      - Missing compliance requirements for the domain
      - Suspiciously low prices (potential quality risk)
      - Unrealistic timelines
      - Vendor qualification concerns
      - Anything else a procurement officer would escalate

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTPUT RULES (non-negotiable):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Return one final JSON with keys: email_meta, context, extracted_values, tag_list,
  sender_notes, review_decisions, validation.
• REMOVE: review_section, all __instruction keys, dspy_audit.
• No markdown fences. No prose before or after the JSON.
• Every previously-empty tag_list value must be filled or set to "__NEEDS_SENDER_INPUT__".
• The JSON must be valid and parseable. No trailing commas.
"""

agent2 = Agent(
    name           = "ProcurementEnricherAgent",
    model          = AZURE_DEPLOYMENT,
    model_settings = ModelSettings(max_tokens=4000, temperature=0.0),
    instructions   = AGENT2_INSTRUCTIONS,
)


# ─────────────────────────────────────────────────────────────────────────────
# Pipeline runner
# ─────────────────────────────────────────────────────────────────────────────

async def run_pipeline(email_key: str, email_text: str) -> dict:
    print(f"\n{'='*60}")
    print(f"Processing: {email_key}")
    print(f"{'='*60}")

    # ── Step 1: Agent1 extracts context + builds tag list ─────────────────────
    print(f"\n[Agent1] Extracting and classifying email content...")
    result1 = await Runner.run(agent1, input=email_text)
    agent1_output = result1.final_output
    print(f"[Agent1 done] {len(agent1_output)} chars")

    # ── Step 2: DSPy audits structure, fills gaps, quarantines noise ──────────
    print(f"[DSPy] Running structural audit...")
    dspy_output = dspy_audit(agent1_output)
    print(f"[DSPy done] {len(dspy_output)} chars")

    # ── Step 3: Agent2 enriches, decides, and validates ───────────────────────
    print(f"[Agent2] Enriching, resolving, and validating...")
    result2 = await Runner.run(agent2, input=dspy_output)
    final_output = result2.final_output
    print(f"[Agent2 done] {len(final_output)} chars")

    # ── Parse + strip any accidental markdown fences ──────────────────────────
    try:
        clean = (
            final_output.strip()
            .removeprefix("```json")
            .removeprefix("```")
            .removesuffix("```")
            .strip()
        )
        parsed = json.loads(clean)
    except json.JSONDecodeError as e:
        print(f"[WARNING] JSON parse failed for {email_key}: {e}")
        parsed = {"raw_output": final_output, "parse_error": str(e)}

    return {
        "email_key"    : email_key,
        "agent1_output": agent1_output,
        "dspy_output"  : dspy_output,
        "final"        : parsed,
    }


async def run_all(emails: dict) -> list:
    results = []
    for key, body in emails.items():
        result = await run_pipeline(key, body)
        results.append(result)
        print(f"\n[FINAL — {key}]")
        print(json.dumps(result["final"], indent=2, ensure_ascii=False), "\n")
    return results


# ─────────────────────────────────────────────────────────────────────────────
# Entry point
# ─────────────────────────────────────────────────────────────────────────────

import json
emails = dict()
with open("/content/bidding_emails.json", encoding="utf-8") as f:
    emails = json.load(f)
print(f"\nLoaded {len(emails)} emails. Starting pipeline...\n")
results = await run_all(emails)
print(f"\n{'='*60}")
print(f"Pipeline complete. {len(results)} emails processed.")
print(f"{'='*60}")


Loaded 10 emails. Starting pipeline...


Processing: email_01

[Agent1] Extracting and classifying email content...
[Agent1 done] 3033 chars
[DSPy] Running structural audit...
[DSPy done] 4381 chars
[Agent2] Enriching, resolving, and validating...
[Agent2 done] 5281 chars

[FINAL — email_01]
{
  "email_meta": {
    "from_address": "sandeep.rao@delta-infra.com",
    "to_address": "bids@acmecorp.com",
    "date_sent": "Monday, May 19, 2026 10:32 AM",
    "subject": "Welding Robots – Looking for a Good Vendor, Thought of You Guys"
  },
  "context": {
    "domain": "manufacturing",
    "request_type": "rfq",
    "summary": "The sender is requesting a quote for 15 units of 6-axis industrial welding robots, including delivery, installation, and AMC pricing, for a new assembly line at their Pune plant.",
    "tone": "formal",
    "relationship": "existing_vendor"
  },
  "extracted_values": {
    "product_name": "6-axis industrial welding robots",
    "quantity": 15,
    "brands_considered": 